In [ ]:
%%capture
!pip install datasets transformers

In [ ]:
from transformers import BertTokenizer

# Machine Learning for NLP

# Building blocks

In this part, we will discover how to build a first neural network for text classification, step by step.

🚧 **TODO** 🚧

Propose a chain of operations that should be applied to the input text, from the input to the output.
When it applies, write the dimension of the expected tensors.

**Answer**

TODO

## Tokenizer

During the last course and during the HW, you used a `WhiteSpaceTokenizer` to tokenize the text.

Here, we are going to use the tokenizer from the model `Bert`, that we will describe later in the course. It comes with the `transformers` library.

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

**Dataset**

We will use the AGNews dataset. It is a dataset with 4 classes: World, Sports, Business, Sci/Tech. We download it from HuggingFace Hub [here](https://huggingface.co/datasets/fancyzhx/ag_news).

In [ ]:
from datasets import load_dataset
import unicodedata
import re

dataset = load_dataset("fancyzhx/ag_news")
dataset = dataset["train"]
dataset = dataset.shuffle(seed=42)
dataset = dataset.select(range(500))
dataset = dataset.train_test_split(test_size=0.3)

print(dataset)


def preprocess_text(text: str) -> str:
    # TODO lower case
    text = text.lower()

    # TODO string normalization.
    text = unicodedata.normalize("NFD", text).encode("ascii", "ignore").decode()

    # TODO remove non alpha numeric characters.
    text = re.sub(r"[^a-z0-9]", " ", text)

    # TODO replace numbers by the <NUM> token.
    text = re.sub(r"\d+", "<NUM>", text)

    # TODO remove double whitespaces.
    text = re.sub(" +", " ", text.strip())
    return text


# Clean the dataset
dataset = dataset.map(lambda x: {"text": preprocess_text(x["text"])})

## Word vectors

The first step since we have access to text in a tokenized form is to use word embeddings.

### Embeddings


🚧 **TODO** 🚧

Experiment with the class `nn.Embedding` from PyTorch.

Construct an embedding layer with:
- `d_embed=50` dimensions
- `n_vocab=V` words in the vocabulary

Then try to get the embedding of the following sentence: `"I love machine learning."`.

In [ ]:
from torch import nn
import torch


embedding_model = # TODO

sentence = "I love machine learning"
tokens = tokenizer(sentence, return_tensors="pt")["input_ids"]
print('Token ids:', tokens)

print('Embedding:')
# TODO

### Aggregation function

Since we need to perform document classfications, we need to aggregate the embeddings into a single vector of size `d`.

### Classification

The last layer of the network should be a linear layer with `c` classes.

🚧 **TODO** 🚧

Write a class named "WordEmbedClassifier" that will take as input a list of ids and return the probability of each class.

In [ ]:
class WordEmbedClassifier(nn.Module):
    def __init__(self, vocab_size, d, n_classes=4):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO

In [ ]:
# Try the model on a simple input:
batch_size = 2
length = 10
model = WordEmbedClassifier(d=5, n_classes=4, vocab_size=10000)
x = torch.randint(0, 10000, (batch_size, length))
output = model(x)
print(output.shape)

## Training

For training we need to iterate over the dataset.

### Data preparation

Here, we will assume that all texts have the same length, using truncation and discarding examples.

🚧 **Question** 🚧

Why do we need to have inputs of the same length?

**Answer**

Because we put them into tensors.

In [ ]:
def tokenize_truncate_and_discard(texts_list, labels_list, tokenizer, length=50):
    new_texts = []
    new_labels = []
    for text, label in zip(texts_list, labels_list):
        tokenized_text = tokenizer.encode(text)
        if len(tokenized_text) < length:
            continue
        new_texts.append(tokenized_text[:length])
        new_labels.append(label)

    return new_texts, new_labels


train_texts, train_labels = tokenize_truncate_and_discard(
    dataset["train"]["text"], dataset["train"]["label"], tokenizer
)
test_texts, test_labels = tokenize_truncate_and_discard(
    dataset["test"]["text"], dataset["test"]["label"], tokenizer
)

train_dataset = [(t, l) for t, l in zip(train_texts, train_labels)]
valid_dataset = [(t, l) for t, l in zip(test_texts, test_labels)]

print("Size before truncating:", len(dataset["train"]["text"]))
print("Size after truncating:", len(train_texts))

Now we need to make batches of examples. We will use the DataLoader class from PyTorch.

🚧 **TODO** 🚧

Load the data into batches. One batch should be a dictionary with the following keys:
- `"input_ids"`: tensor of size (batch_size, L)
- `"labels"`: tensor of size (batch_size,)

In [ ]:
from torch.utils.data import DataLoader


class DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        # `batch` is a list of tuples (text, label)
        texts, labels = zip(*batch)
        # Encode the texts
        return {"input_ids": torch.tensor(texts), "labels": torch.tensor(labels)}


data_collator = DataCollator(tokenizer)

batch_size = 32

train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, collate_fn=# TODO, shuffle=True
)
valid_dataloader = DataLoader(
    valid_dataset, batch_size=batch_size, collate_fn=# TODO, shuffle=True
)
n_valid = len(valid_dataset)
n_train = len(train_dataset)

🔴 **TEST**

In [ ]:
batch = next(iter(train_dataloader))
print(batch)
assert isinstance(batch, dict)
assert "input_ids" in batch
assert "labels" in batch
assert isinstance(batch["input_ids"], torch.Tensor)
assert isinstance(batch["labels"], torch.Tensor)
assert batch["input_ids"].shape[0] == batch_size
assert batch["labels"].shape[0] == batch_size

### Training loop

🚧 **TODO** 🚧

Now write the training loop. Validate the model on the validation set every epoch. Don't forget to plot the learning curves.

In [ ]:
import matplotlib.pyplot as plt
from torch import optim


# TODO
def validation_step(valid_dataloader, model, criterion):
    n_valid = len(valid_dataloader.dataset)
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for batch in valid_dataloader:
            input_ids = batch["input_ids"]
            labels = batch["labels"]
            output = model(input_ids)
            loss = criterion(output, labels)
            total_loss += loss.item()
            correct += # TODO
    return total_loss / n_valid, correct / n_valid


def train_one_epoch(train_dataloader, model, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct = 0
    n_train = len(train_dataloader.dataset)
    for batch in train_dataloader:
        optimizer.zero_grad()
        # TODO

        loss = # TODO
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += # TODO

    return total_loss / n_train, correct / n_train


def train(model, train_dataloader, valid_dataloader, lr=0.01, n_epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    # Track training loss, training accuracy, validation loss and validation accuracy and plot in the end
    train_losses = []
    train_accuracies = []
    valid_losses = []
    valid_accuracies = []

    for epoch in range(n_epochs):
        train_loss, train_accuracy = train_one_epoch(
            train_dataloader, model, optimizer, criterion
        )
        valid_loss, valid_accuracy = validation_step(valid_dataloader, model, criterion)
        print(
            f"Epoch {epoch + 1}: train_loss: {train_loss:.4f}, train_accuracy: {train_accuracy:.4f}, valid_loss: {valid_loss:.4f}, valid_accuracy: {valid_accuracy:.4f}"
        )
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)
        valid_losses.append(valid_loss)
        valid_accuracies.append(valid_accuracy)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="train loss")
    plt.plot(valid_losses, label="valid loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label="train accuracy")
    plt.plot(valid_accuracies, label="valid accuracy")
    plt.legend()


model = WordEmbedClassifier(d=50, n_classes=4, vocab_size=len(tokenizer))

train(model, train_dataloader, valid_dataloader, n_epochs=5)

🚧 **TODO** 🚧

What do you think about the learning curves ?

## Padding and masking

Up to know, we assumed that our texts have the same length. To achieve that, we truncated the texts. However, in practice we want to keep the full texts (up to a given limit, of course)

We will investigate masking and padding to handle texts of different lengths.

🚧 **Question** 🚧

- What will padding achieve?
- What should we be careful about when using padding model?

**Answer**

TODO


In [ ]:
class DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        # Tokenize the texts
        texts = [example["text"] for example in batch]
        labels = [example["label"] for example in batch]
        tokenized_texts = [self.tokenizer.encode(text) for text in texts]
        # Pad the tokenized texts
        max_len = max(len(text) for text in tokenized_texts)
        padded_texts = [
            text + [self.tokenizer.pad_token_id] * (max_len - len(text))
            for text in tokenized_texts
        ]
        pad_mask = [
            [1] * len(text) + [0] * (max_len - len(text)) for text in tokenized_texts
        ]
        return {
            "input_ids": torch.tensor(padded_texts),
            "pad_mask": torch.tensor(pad_mask),
            "labels": torch.tensor(labels),
        }


batch_size = 32
n_train = len(dataset["train"])
n_valid = len(dataset["test"])
data_collator = DataCollator(tokenizer)
train_dataloader = DataLoader(
    dataset["train"], batch_size=batch_size, collate_fn=data_collator, shuffle=True
)
valid_dataloader = DataLoader(
    dataset["test"], batch_size=batch_size, collate_fn=data_collator, shuffle=True
)

In [ ]:
batch = next(iter(train_dataloader))
print(batch)

🚧 **TODO** 🚧

Update the code of the `WordEmbedClassifier` to handle padding.


In [ ]:
class WordEmbedClassifier(nn.Module):
    def __init__(self, vocab_size, d, n_classes=4):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d)
        self.class_projection = nn.Linear(d, n_classes)

    def forward(self, input_ids, pad_mask):
        hidden_states = self.embeddings(input_ids)  # (B, L, d)

        # TODO

        return self.class_projection(hidden_states)

🔴 **TEST**

In [ ]:
model = WordEmbedClassifier(d=50, n_classes=4, vocab_size=10000)

seq_len = 10
x_without_pad = torch.randint(0, 10000, (1, seq_len))
pad_mask = torch.ones(1, seq_len)
out_without_pad = model(x_without_pad, pad_mask)

x_with_pad = torch.randint(0, 10000, (1, 2 * seq_len))
pad_mask = torch.ones(1, 2 * seq_len)
pad_mask[:, seq_len:] = 0
out_with_pad = model(x_with_pad, pad_mask)

assert out_without_pad.shape == out_with_pad.shape

In [ ]:
# TODO
def validation_step(valid_dataloader, model, criterion):
    n_valid = len(valid_dataloader.dataset)
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for batch in valid_dataloader:
            input_ids = batch["input_ids"]
            pad_mask = batch["pad_mask"]
            labels = batch["labels"]
            output = model(input_ids, pad_mask)
            loss = criterion(output, labels)
            total_loss += loss.item()
            correct += (output.argmax(1) == labels).sum().item()
    return total_loss / n_valid, correct / n_valid


def train_one_epoch(train_dataloader, model, optimizer, criterion):
    n_train = len(train_dataloader.dataset)
    model.train()
    total_loss = 0.0
    correct = 0
    for batch in train_dataloader:
        optimizer.zero_grad()
        # TODO

        loss = # TODO
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += # TODO
    return total_loss / n_train, correct / n_train


def train(model, train_dataloader, valid_dataloader, lr=0.01, n_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Track training loss, training accuracy, validation loss and validation accuracy and plot in the end
    train_losses = []
    train_accuracies = []
    valid_losses = []
    valid_accuracies = []

    for epoch in range(n_epochs):
        train_loss, train_accuracy = train_one_epoch(
            train_dataloader, model, optimizer, criterion
        )
        valid_loss, valid_accuracy = validation_step(valid_dataloader, model, criterion)
        print(
            f"Epoch {epoch + 1}: train_loss: {train_loss:.4f}, train_accuracy: {train_accuracy:.4f}, valid_loss: {valid_loss:.4f}, valid_accuracy: {valid_accuracy:.4f}"
        )
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)
        valid_losses.append(valid_loss)
        valid_accuracies.append(valid_accuracy)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="train loss")
    plt.plot(valid_losses, label="valid loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label="train accuracy")
    plt.plot(valid_accuracies, label="valid accuracy")
    plt.legend()

In [ ]:
model = WordEmbedClassifier(
    d=10,
    n_classes=4,
    vocab_size=len(tokenizer),
)
train(
    model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    lr=0.01,
    n_epochs=10,
)

## Sequence processing module

Now we built a simple classifier using word embeddings.

But is our model really different from a simple bag-of-words model?

🚧 **Question** 🚧

How can we make our model more powerful?

**Answer**

TODO

🚧 **Question** 🚧

What kind of module can we use?

**Answer**

TODO

## Convolution1D

Look at the documentation of the Conv1d layer. Read it carefully and try to completely understand the following code. A convolution layer expects a tensor as input, with the following dimensions *(B, D, L)*:
- B: size of the batch, the number of examples (here the number of sequences).
- D: the dimension of the vectors for each time step
- L: the length of the input sequence (the number of tokens in the sequence).

🚧 **Question** 🚧

Is this shape directly compatible with our Embeddings layer defined above?

🚧 **TODO** 🚧

- Make sure the following code computing a convolution run and is consistent.
- Draw what happens to better understand the obtained dimensions.

In [ ]:
d_embed = 50
embedding_layer = nn.Embedding(num_embeddings=len(tokenizer), embedding_dim=d_embed)

sequence_embedding = embedding_layer(batch["input_ids"][:2])
print("Sequence embedding shape:", sequence_embedding.shape)

convolution_layer = nn.Conv1d(in_channels=d_embed, out_channels=d_embed, kernel_size=3)
convolution_output = # TODO
print("Convolution output shape:", convolution_output.shape)

🚧 **TODO** 🚧

Write a class named "Conv1dClassifier" that implements a convolutional neural network for text classification.

In [ ]:
class ConvClassifier(nn.Module):
    def __init__(self, vocab_size, d, n_classes=4):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d)
        self.class_projection = nn.Linear(d, n_classes)
        self.conv1 = # TODO
        self.conv2 = # TODO

    def forward(self, input_ids, pad_mask):
        # TODO


In [ ]:
model = ConvClassifier(
    d=10,
    n_classes=4,
    vocab_size=len(tokenizer),
)
train(
    model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    lr=0.05,
    n_epochs=10,
)